# Day 3 — Band gap from UV-Vis DRS with a Tauc plot (Kubelka–Munk)
**Python for Chemists** series · Muhammad Amjad

Day 1: plot XRD · Day 2: XRD peaks + crystallite size · **Day 3: band gap**.

Every photocatalysis paper reports a band gap (Eg). Most people draw the tangent line by hand in Origin. Today Python will:
1. convert reflectance into absorption with the **Kubelka–Munk** function: $F(R) = \dfrac{(1-R)^2}{2R}$
2. build the **Tauc plot**: $(F(R)\,h\nu)^{1/n}$ vs $h\nu$
3. find the straight (steepest) part **automatically**, fit a line, and read Eg where the line hits zero.

- $h\nu$ (eV) = 1239.84 / wavelength (nm)
- *n* = 2 for an **indirect** allowed transition (exponent 1/2), *n* = 1/2 for a **direct** one (exponent 2)

Sample data: `TiO2_DRS.csv` — *illustrative* (simulated) diffuse reflectance of anatase TiO₂ (indirect gap). Replace it with your own two-column file (wavelength, reflectance as a fraction 0–1).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Step 1 — Read the data and convert wavelength to energy

In [ ]:
# Step 1: read reflectance data and convert wavelength (nm) to photon energy (eV)
df = pd.read_csv("TiO2_DRS.csv")
wl = df["wavelength_nm"].values
R  = df["reflectance"].values        # fraction (0-1). If yours is in %, divide by 100
E  = 1239.84 / wl                    # h*nu in eV

## Step 2 — Kubelka–Munk and the Tauc value
Change `transition` to `"direct"` if your material has a direct band gap.

In [ ]:
# Step 2: Kubelka-Munk function, then the Tauc y-value
F = (1 - R)**2 / (2 * R)

transition = "indirect"              # "indirect" (TiO2 anatase, g-C3N4 often) or "direct"
power = 0.5 if transition == "indirect" else 2
tauc = (F * E)**power

## Step 3 — Find the straight part automatically and extrapolate to zero
The code smooths the curve, finds where it rises most steeply, fits a straight line to the points around that spot, and solves *line = 0* for Eg.

In [ ]:
# Step 3: find the steepest region, fit a line, extrapolate to the energy axis
order = np.argsort(E)                       # sort from low to high energy
E, tauc = E[order], tauc[order]
smooth = np.convolve(tauc, np.ones(9)/9, mode="same")
slope = np.gradient(smooth, E)
i = np.argmax(slope[10:-10]) + 10           # steepest point (ignore the edges)
win = (E > E[i] - 0.12) & (E < E[i] + 0.12) # +/- 0.12 eV around it
m, c = np.polyfit(E[win], tauc[win], 1)     # straight line: y = m*E + c
Eg = -c / m                                 # where the line crosses y = 0
print(f"Band gap Eg = {Eg:.2f} eV  ({transition} transition)")

## Step 4 — Plot

In [ ]:
# Step 4: plot the Tauc curve with the fitted tangent
fig, ax = plt.subplots(figsize=(7, 4.5), dpi=200)
ax.plot(E, tauc, color="#1f3b73", lw=1.5, label="Tauc curve")
xl = np.linspace(Eg, E[win].max() + 0.25, 50)
ax.plot(xl, m*xl + c, "--", color="#d1495b", lw=1.5, label="Linear fit (auto)")
ax.axvline(Eg, color="grey", lw=0.8, ls=":")
ax.annotate(f"Eg = {Eg:.2f} eV", (Eg, 0), xytext=(-75, 60), textcoords="offset points",
            arrowprops=dict(arrowstyle="->"), fontsize=10)
ax.set_xlabel("Photon energy hν (eV)")
ax.set_ylabel(r"$(F(R)\,h\nu)^{1/2}$" if transition == "indirect" else r"$(F(R)\,h\nu)^{2}$")
ax.set_title("TiO$_2$ — automatic Tauc plot band gap")
ax.set_xlim(2.4, 4.6)
ax.set_ylim(bottom=0)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig("day03_tauc.png")
plt.show()

## Important notes for real data
- **Choose the right exponent.** Using the direct formula for an indirect material (or the reverse) shifts Eg. Check the literature for your material.
- **Reflectance as a fraction:** if your instrument gives % reflectance, divide by 100 first.
- **Check the fit window by eye.** Plot the points inside `win`; if they are not on the straight part, change ±0.12 eV.
- Composites with **sub-gap absorption** (e.g. dark ferrites mixed with TiO₂) can give a sloping baseline — then extrapolate to the baseline line, not to zero.
- Eg from a Tauc plot is an estimate (±0.05–0.1 eV); report it with the method used.